# HW5: Bigfoot Reports with Altair
Two plots of the BFRO Bigfoot sightings dataset, saved as vega-lite JSON for my Jekyll page.

In [ ]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

url = 'https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv'
df = pd.read_csv(url)
df.head()

## Plot 1: reports per year with a brush that filters the top states
Drop rows with no date, pull out the year, and keep 1960 onward since earlier years are very sparse.

In [ ]:
df1 = df.dropna(subset=['date']).copy()
df1['year'] = pd.to_datetime(df1['date']).dt.year
df1 = df1[(df1['year'] >= 1960) & (df1['classification'] != 'Class C')]

yearly = df1.groupby(['year', 'classification']).size().reset_index(name='reports')
by_state = df1.groupby(['year', 'state']).size().reset_index(name='reports')

In [ ]:
brush = alt.selection_interval(encodings=['x'])

timeline = alt.Chart(yearly).mark_line(point=True).encode(
    x=alt.X('year:O', title='Year', axis=alt.Axis(values=list(range(1960, 2025, 5)))),
    y=alt.Y('reports:Q', title='Number of reports'),
    color=alt.Color('classification:N', title='Class', scale=alt.Scale(scheme='dark2')),
    tooltip=['year', 'classification', 'reports']
).add_params(brush).properties(width=650, height=250, title='Bigfoot reports per year (drag to select years)')

states = alt.Chart(by_state).mark_bar(color='#1b9e77').encode(
    x=alt.X('sum(reports):Q', title='Reports in selected years'),
    y=alt.Y('state:N', sort='-x', title='State'),
    tooltip=['state', 'sum(reports):Q']
).transform_filter(brush).transform_aggregate(
    reports='sum(reports)', groupby=['state']
).transform_window(
    rank='rank(reports)', sort=[alt.SortField('reports', order='descending')]
).transform_filter(alt.datum.rank <= 15).properties(width=650, height=300, title='Top 15 states for the selected years')

plot1 = alt.vconcat(timeline, states)
plot1

In [ ]:
plot1.save('bigfoot_years_states.json')

## Plot 2: reports by season and class, with a state dropdown
Count reports for each state, season, and class. Unknown season is dropped.

In [ ]:
df2 = df[df['season'] != 'Unknown']
season_counts = df2.groupby(['state', 'season', 'classification']).size().reset_index(name='reports')
all_states = season_counts.groupby(['season', 'classification'])['reports'].sum().reset_index()
all_states['state'] = 'All states'
season_counts = pd.concat([all_states, season_counts])

In [ ]:
state_list = ['All states'] + sorted(df2['state'].unique())
pick = alt.selection_point(
    fields=['state'], value='All states',
    bind=alt.binding_select(options=state_list, name='State: ')
)

plot2 = alt.Chart(season_counts).mark_bar().encode(
    x=alt.X('season:N', sort=['Spring', 'Summer', 'Fall', 'Winter'], title='Season', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('reports:Q', title='Number of reports'),
    color=alt.Color('classification:N', title='Class', scale=alt.Scale(scheme='dark2')),
    tooltip=['state', 'season', 'classification', 'reports']
).add_params(pick).transform_filter(pick).properties(width=500, height=300, title='Bigfoot reports by season')

plot2

In [ ]:
plot2.save('bigfoot_seasons.json')